# Aircraft Damage and Caption Reliability

**Research question:** Can a crack/dent classifier work while a general image captioner still describes the wrong object?

Kerem Ozcan | IBM AI Engineering portfolio study | Prepared with AI assistance

This independently authored experiment applies course concepts. It is not an IBM assessment notebook. Scores below are produced by the current run; no supplied course predictions/checkpoints are used.

## Experiment design

Use all 300 training, 96 validation and 50 test images after exact-pixel duplicate/conflict checks. Freeze ImageNet VGG16, use its expected BGR/channel-centering preprocessing and train a fresh head for ten epochs. Restore the validation-selected head and include remainder batches. Run BLIP separately on one predetermined example per test class; leave human-review fields empty.

**Evaluation boundary:** This is a small teaching dataset with unverified source-photo/near-duplicate overlap. Classification and captioning are separate models, with no damage localization or joint training. Two captions support a qualitative failure analysis, not a caption benchmark or an aircraft safety system.

## Runtime

Kaggle internet access is required for provider-hosted data and public model downloads. Retain Kaggle's preinstalled Torch/TensorFlow builds; the cell pins the Transformers/PEFT interfaces used here. CPU is supported; an enabled GPU accelerates the Torch projects. Actual package versions and source hashes are saved with the metrics.

In [ ]:
import os
import subprocess
import sys
os.environ["USE_TF"] = "0"  # Transformers uses PyTorch; the aircraft module imports TF separately.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.45.2", "peft==0.11.1", "accelerate>=0.30,<2"], check=True)
from pathlib import Path
import json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()/".lab-cache/kaggle-local"
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH = Path("/kaggle/temp/ibm-portfolio") if Path("/kaggle/working").is_dir() else WORK/"scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)


## Data provenance

Data: [Aircraft Damage Detection by Youssef Donia](https://universe.roboflow.com/youssef-donia-fhktl/aircraft-damage-detection-1j9qk), CC BY 4.0, adapted as crack/dent folders by IBM Skills Network. Models: [Keras VGG16](https://keras.io/api/applications/vgg/) and [Salesforce BLIP](https://huggingface.co/Salesforce/blip-image-captioning-base). No source dataset or pretrained weights are bundled.

Expected download SHA-256: `e8cc631b5eb6f373f6e1d47b44cdedd5fb2dd7e6ce7fc28877a1daa4a8f22bc6`. The downloader rejects changed bytes and unsafe tar members.

In [ ]:
"""Small, standalone data acquisition helpers embedded in the Kaggle releases."""

import hashlib
import shutil
import tarfile
import urllib.request
from pathlib import Path


def verified_download(url: str, target: Path, expected_sha256: str) -> Path:
    """Download to a temporary file and accept only the reviewed dataset bytes."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        return target
    temporary = target.with_suffix(target.suffix + ".download")
    try:
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != expected_sha256:
            raise ValueError("Downloaded data changed; review its provenance before running.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target


def unpack_regular_tar(archive: Path, destination: Path) -> None:
    """Reject links, special files and traversal before writing any archive member."""
    root = destination.resolve()
    with tarfile.open(archive) as bundle:
        members = bundle.getmembers()
        for member in members:
            path = root / member.name
            if (
                Path(member.name).is_absolute()
                or not path.resolve().is_relative_to(root)
                or not (member.isfile() or member.isdir())
            ):
                raise ValueError("Unsafe dataset archive member.")
        for member in members:
            path = root / member.name
            if member.isdir():
                path.mkdir(parents=True, exist_ok=True)
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with bundle.extractfile(member) as source, path.open("wb") as target:
                    shutil.copyfileobj(source, target)


In [ ]:
# Fetch from the provider; reject changed archives by SHA-256.
DATA_DIR = SCRATCH / "inputs" / 'aircraft'
DATA_PATH = DATA_DIR / 'aircraft_damage_dataset_v1'
USING_LOCAL_CACHE = False
if not Path("/kaggle/working").is_dir():
    local_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"provenance/source-manifest.json").exists()), None)
    if local_root is not None:
        candidate = local_root / '.lab-cache/02-deep-learning-with-keras/aircraft-damage-classification-and-captioning/aircraft_damage_dataset_v1'
        if candidate.exists():
            DATA_PATH = candidate
            USING_LOCAL_CACHE = True
if not USING_LOCAL_CACHE:
    archive = verified_download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZjXM4RKxlBK9__ZjHBLl5A/aircraft-damage-dataset-v1.tar', DATA_DIR/'aircraft_damage_dataset_v1.tar', 'e8cc631b5eb6f373f6e1d47b44cdedd5fb2dd7e6ce7fc28877a1daa4a8f22bc6')
    if archive.suffix == ".tar":
        unpack_regular_tar(archive, DATA_DIR)
    else:
        DATA_PATH = archive
if not DATA_PATH.exists():
    raise FileNotFoundError("Expected dataset layout was not produced.")
OUTPUT = WORK / 'aircraft'
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = SCRATCH / "runs" / 'aircraft'


## Implementation

Expand this cell to inspect the complete standalone implementation. The split manifests, actual processed sample counts and selected-checkpoint reload checks are part of the experiment.

In [ ]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256 = 'ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256 = '3d442cb44a949540bd46098c8861b9a1f97e7bcc3bd7a758a72a9fd0593b7426'
"""Train a VGG16 feature classifier and inspect separate BLIP descriptions."""


import argparse
import hashlib
import os
import time
from pathlib import Path

import numpy as np


VGG_NOTOP_MD5 = "6d6bbae143d832006294945121d1f1fc"


def vgg_weight_source(weights: Path, allow_downloads: bool) -> str:
    """Use a verified explicit file path so offline loading cannot invoke get_file."""
    if not weights.is_file():
        if allow_downloads:
            return "imagenet"
        raise FileNotFoundError("Cache the ImageNet VGG16 weights or explicitly allow downloads.")
    digest = hashlib.md5(usedforsecurity=False)
    with weights.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != VGG_NOTOP_MD5:
        raise ValueError("Cached VGG16 weights do not match the published Keras checksum.")
    return str(weights)


def rgb_array(path: Path, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Return displayable, unscaled RGB; preprocessing is a separate model operation."""
    from PIL import Image, ImageOps

    with Image.open(path) as image:
        return np.asarray(
            ImageOps.exif_transpose(image).convert("RGB").resize(size), dtype=np.float32
        )


def preprocess_rgb(images: np.ndarray) -> np.ndarray:
    """Apply the pretrained VGG16 contract while preserving RGB display inputs."""
    from tensorflow.keras.applications.vgg16 import preprocess_input

    return preprocess_input(np.asarray(images, dtype=np.float32).copy())


def extract_features(
    root: Path, rows: list[dict], backbone, preprocess, batch_size: int = 8
) -> np.ndarray:
    """Include the final incomplete batch, preserving record order."""
    features = []
    for start in range(0, len(rows), batch_size):
        batch = np.stack(
            [rgb_array(root / row["file"]) for row in rows[start : start + batch_size]]
        )
        features.append(np.asarray(backbone(preprocess(batch.copy()), training=False)))
    if not features:
        raise ValueError("Cannot extract an empty split.")
    return np.concatenate(features)


def caption_image(processor, model, image, max_new_tokens: int = 32) -> str:
    """Run the captioner directly; failures propagate instead of becoming fake captions."""
    import torch

    model.eval()
    inputs = processor(images=image, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()
    if not caption:
        raise ValueError("Captioner returned an empty description.")
    return caption


def run(
    data_root: Path,
    output: Path,
    *,
    full: bool = False,
    allow_downloads: bool = False,
    seed: int = 42,
    epochs: int = 10,
) -> dict:
    """Use the existing source splits after exact-pixel leakage/label checks."""
    os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
    os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
    import tensorflow as tf
    from PIL import Image, ImageOps
    from transformers import BlipForConditionalGeneration, BlipProcessor

    tf.keras.utils.set_random_seed(seed)
    rows = index_images(
        data_root, ["crack", "dent"], {"train": "train", "validation": "valid", "test": "test"}
    )
    rows, removed = clean_image_splits(rows)
    write_json(output / "split-manifest.json", {"records": rows, "removed": removed})
    groups = {
        role: [row for row in rows if row["role"] == role]
        for role in ["train", "validation", "test"]
    }
    if any({r["label"] for r in group} != {0, 1} for group in groups.values()):
        raise ValueError("Each cleaned split must contain both crack and dent images.")
    selected = {
        role: balanced_subset(group, None if full else (8 if role == "train" else 4), seed)
        for role, group in groups.items()
    }
    weights = (
        Path(os.environ.get("KERAS_HOME", str(Path.home() / ".keras")))
        / "models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
    )
    weight_source = vgg_weight_source(weights, allow_downloads)
    started = time.perf_counter()
    backbone = tf.keras.applications.VGG16(include_top=False, weights=weight_source, pooling="avg")
    backbone.trainable = False
    feature_arrays = {
        role: extract_features(data_root, group, backbone, preprocess_rgb)
        for role, group in selected.items()
    }
    labels = {
        role: np.asarray([r["label"] for r in group], dtype=np.float32)
        for role, group in selected.items()
    }
    head = tf.keras.Sequential(
        [
            tf.keras.layers.Input((512,)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(1),
        ]
    )
    head.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    )
    history, best_loss = [], float("inf")
    checkpoint = output / "best-classifier.keras"
    rng = np.random.default_rng(seed)
    for epoch in range(epochs if full else 2):
        order = rng.permutation(len(selected["train"]))
        for start in range(0, len(order), 8):
            indices = order[start : start + 8]
            head.train_on_batch(feature_arrays["train"][indices], labels["train"][indices, None])
        logits = head(feature_arrays["validation"], training=False)
        loss = float(
            tf.keras.losses.binary_crossentropy(
                labels["validation"][:, None], logits, from_logits=True
            )
            .numpy()
            .mean()
        )
        history.append({"epoch": epoch + 1, "validation_loss": loss})
        if loss < best_loss:
            best_loss = loss
            head.save(checkpoint)
    head = tf.keras.models.load_model(checkpoint)
    expected = np.asarray(head(feature_arrays["validation"], training=False))
    reloaded = tf.keras.models.load_model(checkpoint)
    np.testing.assert_array_equal(
        expected, np.asarray(reloaded(feature_arrays["validation"], training=False))
    )
    probabilities = (
        tf.sigmoid(head(feature_arrays["test"], training=False)).numpy().ravel().tolist()
    )
    metrics = classification_metrics(labels["test"].astype(int).tolist(), probabilities)
    model_id = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(model_id, local_files_only=not allow_downloads)
    captioner = BlipForConditionalGeneration.from_pretrained(
        model_id, local_files_only=not allow_downloads
    )
    captions = []
    for label in [0, 1]:
        index = next(i for i, row in enumerate(selected["test"]) if row["label"] == label)
        row = selected["test"][index]
        with Image.open(data_root / row["file"]) as image:
            text = caption_image(
                processor, captioner, ImageOps.exif_transpose(image).convert("RGB")
            )
        captions.append(
            {
                "file": row["file"],
                "label": ["crack", "dent"][label],
                "classifier_prediction": ["crack", "dent"][int(probabilities[index] >= 0.5)],
                "caption": text,
                "visible_defect_supported": None,
                "hallucination_review": None,
            }
        )
    report = {
        "project": "aircraft",
        "profile": "full" if full else "bounded-pilot",
        "seed": seed,
        "class_names": ["crack", "dent"],
        "complete_split_counts": split_counts(rows),
        "selected_split_counts": split_counts([r for group in selected.values() for r in group]),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "vgg_weights_sha256": file_sha256(weights),
        "preprocessing": "RGB 0..255 -> VGG BGR channel centering; no 1/255 rescale",
        "checkpoint_sha256": file_sha256(checkpoint),
        "exact_reload": True,
        "test": {**metrics, "ids": [r["file"] for r in selected["test"]]},
        "history": history,
        "caption_model": model_id,
        "caption_revision": getattr(captioner.config, "_commit_hash", None),
        "captions": captions,
        "caption_quality": "Not evaluated; human review fields remain null.",
        "near_duplicate_or_source_group_leakage_audited": False,
        "seconds": time.perf_counter() - started,
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
        "runtime": runtime(["tensorflow", "transformers", "torch", "numpy", "Pillow"]),
    }
    write_json(output / "metrics.json", report)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument(
        "--output", type=Path, default=Path(".lab-cache/selected-projects/aircraft")
    )
    parser.add_argument("--full", action="store_true")
    parser.add_argument("--allow-downloads", action="store_true")
    args = parser.parse_args()
    report = run(args.data_root, args.output, full=args.full, allow_downloads=args.allow_downloads)
    print(report["test"])



In [ ]:
report = run(DATA_PATH, RUN_OUTPUT, full=True, seed=42, allow_downloads=True)
for filename in ['metrics.json', 'split-manifest.json']:
    shutil.copyfile(RUN_OUTPUT/filename, OUTPUT/filename)
print('Executed profile:', report['profile'])
print('Evaluated split sizes:', report['selected_split_counts'])
print('Runtime:', report['runtime'])

## Measured results

Read the sample counts and stated limitations alongside these measurements.

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(11,5))
for ax, row in zip(axes, report["captions"]):
    with Image.open(DATA_PATH/row["file"]) as image: ax.imshow(image.convert("RGB"))
    ax.set_title(f"True: {row['label']} | classifier: {row['classifier_prediction']}")
    ax.set_xlabel(row["caption"], wrap=True); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Inspect the image against the caption; human-review fields remain empty")
fig.tight_layout(); fig.savefig(OUTPUT/"results.png", dpi=160); plt.show()
print("Complete classifier test:", report["test"])
print("Classifier reload:", report["exact_reload"], "caption quality:", report["caption_quality"])


## What this experiment establishes

The saved metrics bind this execution to its source hashes, dataset partitions and runtime versions. Checkpoint reload is an implementation check. It does not establish task quality by itself.

This is a small teaching dataset with unverified source-photo/near-duplicate overlap. Classification and captioning are separate models, with no damage localization or joint training. Two captions support a qualitative failure analysis, not a caption benchmark or an aircraft safety system.

## Attribution

Data: [Aircraft Damage Detection by Youssef Donia](https://universe.roboflow.com/youssef-donia-fhktl/aircraft-damage-detection-1j9qk), CC BY 4.0, adapted as crack/dent folders by IBM Skills Network. Models: [Keras VGG16](https://keras.io/api/applications/vgg/) and [Salesforce BLIP](https://huggingface.co/Salesforce/blip-image-captioning-base). No source dataset or pretrained weights are bundled.

Concepts studied through the [IBM AI Engineering Professional Certificate](https://www.coursera.org/professional-certificates/ai-engineer). Source/data/model licenses remain with their providers. No general license is applied to IBM course materials or third-party datasets.